In [0]:
USE CATALOG gov_transparencia;
USE SCHEMA gold;

CREATE OR REPLACE TABLE gold_series_temporais_estabelecimentos
USING DELTA
AS 

WITH base AS (
SELECT 
    XXHASH64(TRIM(UPPER(cartao_corporativo.estabelecimento_nome))) AS id_hash_estabelecimento,
    cartao_corporativo.estabelecimento_nome,
    cartao_corporativo.estabelecimento_id,
    cartao_corporativo.estabelecimento_cnpj,
    cartao_corporativo.estabelecimento_tipo,
    DATE_TRUNC('month', data_transacao) AS mes,
    YEAR(data_transacao) AS ano,
    cartao_corporativo.valor_transacao,

    CASE MONTH(data_transacao)
      WHEN 1 THEN 'Janeiro'
      WHEN 2 THEN 'Fevereiro'
      WHEN 3 THEN 'Março'
      WHEN 4 THEN 'Abril'
      WHEN 5 THEN 'Maio'
      WHEN 6 THEN 'Junho'
      WHEN 7 THEN 'Julho'
      WHEN 8 THEN 'Agosto'
      WHEN 9 THEN 'Setembro'
      WHEN 10 THEN 'Outubro'
      WHEN 11 THEN 'Novembro'
      WHEN 12 THEN 'Dezembro'
    END AS mes_nome

FROM silver.cartao_corporativo
)

SELECT 
    id_hash_estabelecimento,
    estabelecimento_nome,
    estabelecimento_tipo,
    mes,
    ano,
    mes_nome,
    CAST(SUM(base.valor_transacao) AS DECIMAL(18,2)) AS total_mes,
    COUNT(*) AS qtd_transacoes_mes,
    CAST(AVG(base.valor_transacao) AS DECIMAL(18,2)) AS media_mes

FROM base

GROUP BY estabelecimento_nome,estabelecimento_tipo,mes,mes_nome,id_hash_estabelecimento, ano